This notebook demonstrates how sampling affects waveform representations and the Discrete Fourier Transform (DFT).

It is intended mainly for educational use, showcasing basic waveform generation and simple interactions (including AM and FM).

Key things to explore:
- Visualizations of sampling/Nyquist effects (aligned grids between the waveform and DFT views)
- Effects of window functions on DFT analysis
- Effects of sample rate (SR) and sample length on time-domain waveforms and DFT results

In [1]:
import numpy as np
from audiospylt.waveform_utils import generate_waveforms, plot_waveforms


config = {
    # Time step in seconds (delta time)
    'dt': 0.001,

    # Total duration of the waveform
    't_max': 2.0,

    # selected_waveforms (list[str]): which generated signals to plot/DFT.
    # Allowed values (currently implemented in audiospylt.waveform_utils.generate_waveforms):
    # - 'sine1': amp1 * sin(2π*freq1*t + phase1)
    # - 'sine2': amp2 * sin(2π*freq2*t + phase2)
    # - 'square': amp1 * scipy.signal.square(2π*freq1*t + phase1, duty=square_duty)
    # - 'saw': amp1 * scipy.signal.sawtooth(2π*freq1*t + phase1, width=saw_width)
    # - 'am': (amp1 + index_am*sine2(t)) * sin(2π*freq1*t + phase1)
    # - 'fm': frequency deviation model with phase integration:
    #         f_inst(t) = freq1 + index_fm*sine2(t)
    #         phase(t) = phase1 + 2π * ∫ f_inst(t) dt
    # - 'sum': sine1 + sine2
    # - 'comb': time-domain splice of sine1 and sine2 in ~N/2-sample blocks.
    #          With the current code this is effectively: first half of samples from sine1,
    #          then second half from sine2 (block length ≈ round(t_max/(2*dt))).
    'selected_waveforms': ['saw'], 

    # Parameters for sine wave 1: amp, freq, phase
    'amp1': 1.0, 'freq1': 112.5, 'phase1': 0*np.pi,

    # Parameters for sine wave 2: amp, freq, phase
    'amp2': 1.0, 'freq2': 17, 'phase2': 1*np.pi,

    # Set the modulation index for AM
    'index_am': 1,

    # Set the modulation index for FM.
    # In the current implementation, index_fm scales the deviation term in:
    #   f_inst(t) = freq1 + index_fm*sine2(t)
    # then phase is obtained by discrete integration of f_inst.
    'index_fm': 0.1,

    # square_duty (float in [0, 1]): duty cycle for 'square' waveform.
    'square_duty': 0.5,

    # saw_width (float in [0, 1]): shape parameter for 'saw' waveform (0.5 gives triangle).
    'saw_width': 0.2,

    # Apply window function (handled inside generate_waveforms)
    'apply_window': True,

    # Type of window function to apply. Reference here: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.get_window.html
    'window_type': 'tukey',

    # Add dotted lines overlay (disable if using high SR!)
    'add_dotted_lines': False,
}

waveform_data, t = generate_waveforms(config)

plot_waveforms(
    waveform_data,
    t,
    config,
    plot_width=1100,
    plot_height=600,
)


Sampling rate: 1000.0 Hz
Nyquist frequency: 500.0 Hz
Total number of available sample points: 2000
Frequency Resolution: 0.5 Hz


Save or preview your audio output here:

In [2]:
from audiospylt.generate_wave_file import render_selected_waveforms

player = True
save_audio = False

# Renders a single stream by mixing ALL waveforms in config['selected_waveforms']
# Assign to a variable (or use a trailing ';') to avoid Jupyter printing the returned tuple.

_result = render_selected_waveforms(
    waveform_data,
    config,

    # mix_mode:
    # - "sum"  -> add selected waveforms (export path normalizes anyway)
    # - "mean" -> average selected waveforms (useful if you select many signals)
    mix_mode="sum",

    # fs_target_name: export sample rate. Use "source" to keep the original rate (= 1/dt).
    # Examples: "44.1kHz", "48kHz", "96kHz", "source", or an int Hz (e.g. 44100)
    fs_target_name="44.1kHz",

    # bit_rate: PCM bit depth (supported: 16 or 24)
    bit_rate=24,

    # filename_template: used only when save_audio=True; placeholders:
    # {fs_target_name}, {bit_rate}, {timestamp}, {timestamp_format}
    filename_template="testing_{fs_target_name}_{bit_rate}bit_{timestamp}",

    # timestamp_format: strftime format for {timestamp}
    timestamp_format="%Y-%m-%d_%H-%M-%S",

    # output_dir: where to save the .wav when save_audio=True.
    # - None -> defaults to ./rendered_audio under the current working directory
    # - str/Path -> writes into that folder (created if it doesn't exist)
    # output_dir=r"C:\\Users\\egorp\\Desktop\\audiospylt_renders",

    # save_audio:
    # - True  -> writes .wav and returns a filepath
    # - False -> returns (audio_data, fs_target)
    save_audio=save_audio,

    # player: show an IPython Audio widget
    player=player,

    # sanitize: replace NaN/Inf with finite values before exporting
    sanitize=True,

    # verbose: print warnings/info (missing waveforms, length mismatch trimming, dt rounding)
    verbose=True,
)
